In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *

In [2]:
gamma_1=gamma_x=np.array([[0.,0.,0.,1j],[0.,0.,1j,0.],[0.,-1j,0.,0.],[-1j,0.,0.,0.]])
gamma_2=gamma_y=np.array([[0.,0.,0.,1.],[0.,0.,-1.,0.],[0.,-1.,0.,0.],[1.,0.,0.,0.]])
gamma_3=gamma_z=np.array([[0.,0.,1j,0.],[0.,0.,0.,-1j],[-1j,0.,0.,0.],[0.,1j,0.,0.]])
gamma_4=gamma_t=np.array([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,-1.,0.],[0.,0.,0.,-1.]])
gamma_5=(gamma_1@gamma_2@gamma_3@gamma_4)

CG1=1j*gamma_2@gamma_4@gamma_1
CG2=1j*gamma_2@gamma_4@gamma_2
CG3=1j*gamma_2@gamma_4@gamma_3
CG4=1j*gamma_2@gamma_4@gamma_4
CG5=1j*gamma_2@gamma_4@gamma_5

Lx, Ly, Lz = (48,48,48)
def momentum_phase(mom, sgn):
    nx, ny, nz = mom
    x = np.arange(Lx)[:, None, None]
    y = np.arange(Ly)[None, :, None]
    z = np.arange(Lz)[None, None, :]
    angle = 2 * np.pi * ( nx * x / Lx + ny * y / Ly + nz * z / Lz)
    return np.exp(sgn * 1j * angle)
def momentum_phase_source(mom, src):
    nx,ny,nz=mom
    x,y,z=src
    angle=2 * np.pi * ( nx * x / Lx + ny * y / Ly + nz * z / Lz)
    return np.exp(1j * angle)

In [3]:
eps = np.zeros((3, 3, 3), dtype=np.int8)
eps[0, 1, 2] = eps[1, 2, 0] = eps[2, 0, 1] =  1
eps[2, 1, 0] = eps[1, 0, 2] = eps[0, 2, 1] = -1

src=[38,35,44]

def T1(g1,g2,q1,q2,q3,mom):
    phase=momentum_phase(mom,-1)
    return -np.einsum('abc,lmn,pq,rs,opalzyx,rqbmzyx,stcnzyx,xyz->ot',eps,eps,g1,g2,q1,q2,q3,phase,optimize=True)
def T2(g1,g2,q1,q2,q3,mom):
    phase=momentum_phase(mom,-1)
    return -np.einsum('abc,lmn,sp,qr,otalzyx,qpbmzyx,rscnzyx,xyz->ot',eps,eps,g1,g2,q1,q2,q3,phase,optimize=True)

In [4]:
time_f=4
basepath='/capstor/store/cscs/userlab/lp139/lyan/code/scratch/run/testYan/run/a0000/'
def convert(t):
    return t[...,0]+1j*t[...,1]

src=(38,35,44)

with h5py.File(f'{basepath}propUSS.h5') as fu, h5py.File(f'{basepath}propDSS.h5') as fd, \
    h5py.File(f'{basepath}seqProp_UU.h5') as fsuu, h5py.File(f'{basepath}seqProp_DD.h5') as fsdd,\
    h5py.File(f'{basepath}Diagram_N_0000_sx38sy35sz44st000.h5') as fN, h5py.File(f'{basepath}T_DJN.h5') as fT:
        tu=convert(fu['data'][:,:,:,:,time_f])
        td=convert(fd['data'][:,:,:,:,time_f])
        tsuu=convert(fsuu['data'][:,:,:,:,time_f])
        tsdd=convert(fsdd['data'][:,:,:,:,time_f])
        
        moms_N=fN['/sx38sy35sz44st00/mvec'][:]
        dic_N={}
        for i,mom in enumerate(moms_N):
            dic_N[tuple(mom)] = i
        tN=convert(fN['/sx38sy35sz44st00/NP'][time_f,:,0])
        
        moms_T=fT['/sx00sy00sz00st00/T_DJN/mvec'][:]
        dic_T={}
        for i,mom in enumerate(moms_N):
            dic_T[tuple(mom)] = i
        key2tT={}
        for key in fT['/sx00sy00sz00st00/T_DJN/'].keys():
            if key=='mvec':
                continue
            key2tT[key] = convert(fT[f'/sx00sy00sz00st00/T_DJN/{key}'][time_f,:,0])

In [5]:
mom=(1,-1,-1)
t=(T1(CG5.T,CG5.T,tu,td,tu,mom)+T2(CG5.T,CG5.T,tu,td,tu,mom))*(-1)*momentum_phase_source(mom, src) # (-1) from source N sign
dt=np.reshape(tN[dic_N[mom],:],(4,4))-t
print(np.sum(np.abs(dt)),np.sum(np.abs(t)))

9.982207724998416e-16 1.0150578171921826e-08


In [6]:
mom=(-1,-1,1)
for icg,cgi in enumerate([CG1, CG2, CG3]):
    print(icg)
    
    t=T1(CG5,cgi,tsdd,tu,tu,mom)
    dt=key2tT['T1_DD_U_U'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T1(CG5,cgi,td,tsuu,tu,mom)
    dt=key2tT['T1_D_UU_U'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T1(CG5,cgi,td,tu,tsuu,mom)
    dt=key2tT['T1_D_U_UU'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T1(CG5,cgi,tsuu,td,tu,mom)
    dt=key2tT['T1_UU_D_U'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T1(CG5,cgi,tu,tsdd,tu,mom)
    dt=key2tT['T1_U_DD_U'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T1(CG5,cgi,tu,td,tsuu,mom)
    dt=key2tT['T1_U_D_UU'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T2(CG5,cgi,tsuu,tu,td,mom)
    dt=key2tT['T2_UU_U_D'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T2(CG5,cgi,tu,tsdd,tu,mom)
    dt=key2tT['T2_U_DD_U'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))
    
    t=T2(CG5,cgi,tu,tsuu,td,mom)
    dt=key2tT['T2_U_UU_D'][dic_T[mom],icg]-t
    print(np.sum(np.abs(dt)),np.sum(np.abs(t)))


0


2.579907523180838e-15 1.0341003686684478e-08
2.2505208758231385e-15 8.051094730829507e-09
2.6296480104093423e-15 8.507607793261131e-09
3.109073630373972e-15 1.0717692377332742e-08
2.9419071394824187e-15 1.1536441644924133e-08
2.042669002686177e-15 1.0943451087120643e-08
2.532846327791052e-15 7.539817613888397e-09
2.3870311412367815e-15 7.051134346484025e-09
2.386607343120008e-15 8.436234113866236e-09
1
3.576174430671478e-15 1.2861741168126411e-08
2.811721131811535e-15 9.491748623238768e-09
2.475050307093578e-15 8.974705754805872e-09
2.7733779886198828e-15 1.049543348357468e-08
2.6262781296475504e-15 9.53702978804511e-09
2.59495811137282e-15 8.818693108700869e-09
2.0461135184289553e-15 6.296173759513496e-09
1.728996223139719e-15 6.55707497030899e-09
1.895425625789041e-15 7.073736270234989e-09
2
2.724679164255662e-15 1.0250107524048541e-08
2.08311671873816e-15 7.521677022240937e-09
2.6887926815556664e-15 9.323939472738974e-09
3.3249366373197194e-15 1.0390501622695669e-08
2.79847129311619